In [1]:
! pip install pytorch-forecasting pytorch-lightning


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [2]:
import warnings
import os
import pandas as pd
import numpy as np
import torch
import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from lightning.pytorch.tuner import Tuner
import matplotlib.pyplot as plt

from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer, Baseline
from pytorch_forecasting.data import NaNLabelEncoder
from pytorch_forecasting.data.encoders import MultiNormalizer, TorchNormalizer
from pytorch_forecasting.metrics import QuantileLoss, MAE, SMAPE
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")


torch.set_default_dtype(torch.float32)

# Constants
context_length = 300
forecast_length = 50
target_columns = [
    'COOLANT_TEMPERATURE ()',
    #'ENGINE_RPM ()',
    #'VEHICLE_SPEED ()',
    #'THROTTLE ()',
    #'ENGINE_LOAD ()',
    #'INTAKE_MANIFOLD_PRESSURE ()',
]
time_col = 'ENGINE_RUN_TINE ()'
path = "/Users/darenpalmer/Desktop/UCL/CS/fyp.nosync/data/carOBD/obdiidata"

In [3]:
df_list = []
for file in os.listdir(path):
    if file.endswith('.csv'):
        df = pd.read_csv(f'{path}/{file}', index_col=False)
        df['drive_id'] = file
        df_list.append(df)

print(f'{len(df_list)} files loaded out of {len([f for f in os.listdir(path) if f.endswith(".csv")])}')

129 files loaded out of 129


In [4]:
def remove_zero_variance_columns(df: pd.DataFrame, exclude_cols: list[str] = None) -> pd.DataFrame:
    """
    Compute std of each std-computable column (numeric only)
    """
    if exclude_cols is None:
        exclude_cols = []
    
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    cols_to_check = [col for col in numeric_cols if col not in exclude_cols]
  
    std_df = df[cols_to_check].std()
    zero_variance_cols = std_df[std_df == 0].index.tolist()
  
    print(f'{len(zero_variance_cols)} columns with zero variance: {zero_variance_cols}')
  
    if len(zero_variance_cols) > 0:
        df = df.drop(columns=zero_variance_cols)
  
    return df

In [5]:
def mean_fill_missing_timestamps_and_remove_duplicates(df: pd.DataFrame, time_col: str, id_cols: list[str] = None) -> pd.DataFrame:
    """
    Remove duplicate timestamps by averaging all numeric columns for each unique timestamp.
    This preserves the overall statistics while removing duplicate entries.
    
    Note: The time column itself is not averaged (it becomes the group key).
    Only numeric columns are averaged when multiple rows share the same timestamp.
    """
    if id_cols is None:
        id_cols = []
    
    existing_id_cols = [col for col in id_cols if col in df.columns]
    
    group_cols = [time_col] + existing_id_cols
    
    agg_dict = {}
    for col in df.columns:
        if col not in group_cols:
            if pd.api.types.is_numeric_dtype(df[col]):
                agg_dict[col] = 'mean'
            else:
                agg_dict[col] = 'first'
  
    df_clean = df.groupby(group_cols, as_index=False).agg(agg_dict)
  
    return df_clean

In [6]:
def downsample(df, time_col, source_file_col, downsample_factor=2):
    result_dfs = []
    
    for source_file in df[source_file_col].unique():
        file_df = df[df[source_file_col] == source_file].copy()
        
        if len(file_df) < downsample_factor * 2:
            continue
        
        file_df = file_df.sort_values(time_col).reset_index(drop=True)
        
        # Simple decimation without pre-smoothing
        downsampled = file_df.iloc[::downsample_factor].copy()
        downsampled[time_col] = np.arange(len(downsampled)) * downsample_factor
        
        result_dfs.append(downsampled.reset_index(drop=True))
    
    return pd.concat(result_dfs, ignore_index=True)

In [7]:
def filter_long_drives(df, id_col='drive_id', min_length=608):
    """Keep only drives long enough for your context window"""
    drive_lengths = df.groupby(id_col).size()
    valid_drives = drive_lengths[drive_lengths >= min_length].index
    
    print(f"Keeping {len(valid_drives)}/{df[id_col].nunique()} drives")
    print(f"Dropped {len(df) - df[df[id_col].isin(valid_drives)].shape[0]} timesteps")
    
    return df[df[id_col].isin(valid_drives)].reset_index(drop=True)

In [8]:
# Combine all dataframes
data = pd.concat(df_list, ignore_index=True)

# Clean up
print(f"Total samples: {len(data):,}")
print(f"Unique drives: {data['drive_id'].nunique()}")

# remove some useless columns
data = data.drop(columns=['WARM_UPS_SINCE_CODES_CLEARED ()', 'TIME_SINCE_TROUBLE_CODES_CLEARED ()'])

data = mean_fill_missing_timestamps_and_remove_duplicates(data, time_col=time_col, id_cols=["drive_id"])
data = remove_zero_variance_columns(data, exclude_cols=["drive_id"])
data = downsample(
    data,
    time_col=time_col,
    source_file_col='drive_id',
    downsample_factor=1
)

data = filter_long_drives(data, min_length=context_length + forecast_length)

Total samples: 304,299
Unique drives: 129
3 columns with zero variance: ['FUEL_AIR_COMMANDED_EQUIV_RATIO ()', 'TIME_RUN_WITH_MIL_ON ()', 'DISTANCE_TRAVELED_WITH_MIL_ON ()']
Keeping 106/129 drives
Dropped 6490 timesteps


In [9]:
def add_cross_channel_features(data, target_columns):
    """
    Engineer features that capture cross-channel relationships.
    Add these as conditional columns.
    """
    # RPM-to-Speed ratio (gear indicator)
    if 'ENGINE_RPM ()' in data.columns and 'VEHICLE_SPEED ()' in data.columns:
        data['RPM_SPEED_RATIO'] = data['ENGINE_RPM ()'] / (data['VEHICLE_SPEED ()'] + 1)
    
    # Throttle-to-Load ratio (efficiency indicator)
    if 'THROTTLE ()' in data.columns and 'ENGINE_LOAD ()' in data.columns:
        data['THROTTLE_LOAD_RATIO'] = data['THROTTLE ()'] / (data['ENGINE_LOAD ()'] + 1)
    
    # Speed-based categories
    if 'VEHICLE_SPEED ()' in data.columns:
        data['IS_IDLE'] = (data['VEHICLE_SPEED ()'] < 5).astype(float)
        data['IS_HIGHWAY'] = (data['VEHICLE_SPEED ()'] > 60).astype(float)
    
    # RPM acceleration
    if 'ENGINE_RPM ()' in data.columns:
        data['RPM_ACCEL'] = data.groupby('drive_id')['ENGINE_RPM ()'].diff().fillna(0)
    
    return data

# Apply cross-channel features before preprocessing
data = add_cross_channel_features(data, target_columns)
print("Added cross-channel features")

Added cross-channel features


In [10]:
import numpy as np
import pandas as pd

# Ensure data is sorted
data = data.sort_values(["drive_id", time_col]).reset_index(drop=True)

# Get unique drive IDs
unique_drives = data['drive_id'].unique()
n_drives = len(unique_drives)

# Split drives into train/val/test groups (70/15/15)
train_drives = unique_drives[:int(0.70 * n_drives)]
val_drives   = unique_drives[int(0.70 * n_drives):int(0.85 * n_drives)]
test_drives  = unique_drives[int(0.85 * n_drives):]

print(f"Train drives: {len(train_drives)}, Val drives: {len(val_drives)}, Test drives: {len(test_drives)}")

# Create split dataframes
train_data = data[data['drive_id'].isin(train_drives)].copy()
val_data   = data[data['drive_id'].isin(val_drives)].copy()
test_data  = data[data['drive_id'].isin(test_drives)].copy()

print(f"Train shape: {train_data.shape}, Val shape: {val_data.shape}, Test shape: {test_data.shape}")

Train drives: 74, Val drives: 16, Test drives: 16
Train shape: (47832, 28), Val shape: (18384, 28), Test shape: (9701, 28)


In [13]:
from pytorch_forecasting.data.encoders import NaNLabelEncoder

real_cols = [
    c for c in data.columns
    if c not in ["drive_id", "time_idx"] and c not in target_columns 
]

time_varying_unknown_reals = target_columns + real_cols

time_varying_known_reals = time_col

# Explicit multi-target normalizer
target_normalizer = MultiNormalizer(
    [TorchNormalizer(method="standard", center=True, transformation=None)] * len(target_columns)
)

drive_encoder = NaNLabelEncoder(add_nan=True).fit(data["drive_id"])

training_dataset = TimeSeriesDataSet(
    data=train_data,
    time_idx=time_col,
    target=target_columns,                  
    group_ids=["drive_id"],               
    max_encoder_length=context_length,
    max_prediction_length=forecast_length,
    time_varying_known_reals=time_varying_known_reals,
    time_varying_unknown_reals=time_varying_unknown_reals,
    static_categoricals=None,             
    static_reals=None,
    target_normalizer=target_normalizer,
    categorical_encoders={"drive_id": drive_encoder},
    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True,
)

validation_dataset = TimeSeriesDataSet.from_dataset(
    training_dataset,
    data=val_data,
    stop_randomization=True,
)

test_dataset = TimeSeriesDataSet.from_dataset(
    training_dataset,
    data=test_data,
    stop_randomization=True,
)

batch_size = 64

train_dataloader = training_dataset.to_dataloader(
    train=True,
    batch_size=batch_size,
    num_workers=0,
)

val_dataloader = validation_dataset.to_dataloader(
    train=False,
    batch_size=batch_size,
    num_workers=0,
)

test_dataloader = test_dataset.to_dataloader(
    train=False,
    batch_size=batch_size,
    num_workers=0,
)

In [14]:
pl.seed_everything(42)

accelerator = "mps"

tft = TemporalFusionTransformer.from_dataset(
    training_dataset,
    learning_rate=1e-3,
    hidden_size=64,
    attention_head_size=4,
    dropout=0.1,
    hidden_continuous_size=32,
    loss=QuantileLoss(),
    log_interval=50,
    log_val_interval=1,
    reduce_on_plateau_patience=3,
)

print(f"Number of parameters in model: {tft.size()/1e3:.1f}k")

trainer = pl.Trainer(
    max_epochs=30,
    accelerator=accelerator,
    gradient_clip_val=0.1,
    enable_checkpointing=True,
    callbacks=[
        EarlyStopping(monitor="val_loss", min_delta=1e-4, patience=10, verbose=False, mode="min")
    ],
)

trainer.fit(
    tft,
    train_dataloaders=train_dataloader,
    val_dataloaders=val_dataloader
)

best_tft = TemporalFusionTransformer.load_from_checkpoint(
    trainer.checkpoint_callback.best_model_path,
    weights_only=False
)

Seed set to 42


AssertionError: multiple targets require loss to be MultiLoss but found QuantileLoss(quantiles=[0.02, 0.1, 0.25, 0.5, 0.75, 0.9, 0.98])

In [ ]:
res = Tuner(trainer).lr_find(
    net,
    train_dataloaders=train_dataloader,
    val_dataloaders=val_dataloader,
    min_lr=1e-5,
    max_lr=1e0,
    early_stop_threshold=100,
)
print(f"suggested learning rate: {res.suggestion()}")
fig = res.plot(show=True, suggest=True)
net.hparams.learning_rate = res.suggestion()

In [ ]:
early_stop_callback = EarlyStopping(monitor="val_loss", min_delta=1e-4, patience=10, verbose=False, mode="min")
trainer = pl.Trainer(
    max_epochs=30,
    accelerator=accelerator,
    enable_model_summary=True,
    gradient_clip_val=0.1,
    callbacks=[early_stop_callback],
    limit_train_batches=50,
    enable_checkpointing=True,
)


net = DeepAR.from_dataset(
    training_dataset,
    learning_rate=1e-2,
    log_interval=10,
    log_val_interval=1,
    hidden_size=30,
    rnn_layers=2,
    optimizer="Adam",
    loss=MultivariateNormalDistributionLoss(rank=30),
)

trainer.fit(
    net,
    train_dataloaders=train_dataloader,
    val_dataloaders=val_dataloader,
)

best_net = DeepAR.load_from_checkpoint(
    trainer.checkpoint_callback.best_model_path,
    weights_only=False
)


In [ ]:
import pickle

with open("tft.pkl",'wb') as f:
    pickle.dump(best_net)

In [ ]:
import pickle
best_net = pickle.load(open("tft.pkl",'rb'))

In [ ]:
import matplotlib.pyplot as plt
from pytorch_forecasting.models.temporal_fusion_transformer import TemporalFusionTransformer

best_net = best_net.to(accelerator)

raw_predictions = best_net.predict(test_dataloader, mode="raw", return_x=True, trainer_kwargs=dict(accelerator=accelerator))

for i, target_name in enumerate(target_columns):
    fig, ax = plt.subplots(figsize=(10, 4))
    
    sample_idx = 0
    
    encoder_target = raw_predictions.x["encoder_target"][sample_idx, :, i].cpu().numpy()
    decoder_target = raw_predictions.x["decoder_target"][sample_idx, :, i].cpu().numpy()
    
    prediction = raw_predictions.output["prediction"][sample_idx, :, i].cpu().numpy()
    
    history_len = len(encoder_target)
    ax.plot(range(-history_len, 0), encoder_target, label="History", color="blue")
    ax.plot(range(0, len(decoder_target)), decoder_target, label="Actual", color="green")
    ax.plot(range(0, len(prediction)), prediction, label="Predicted", color="orange")
    
    ax.axvline(0, color="black", linestyle="--", alpha=0.5)
    ax.set_title(f"{target_name}")
    ax.set_xlabel("Time index")
    ax.legend()
    plt.tight_layout()
    plt.show()
